In [20]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

# Retrieve API key securely from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [7]:
from pathlib import Path

# Define project directory in Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/INFS 348 Final Project/INFS-348-Final-Project")

print("Project folder exists:", PROJECT_ROOT.exists())
print("Project location:", PROJECT_ROOT)

Project folder exists: True
Project location: /content/drive/MyDrive/Colab Notebooks/INFS 348 Final Project/INFS-348-Final-Project


In [8]:
required_files = [
    "SRC/Agent/prediction_tool.py",
    "SRC/Agent/explanation_tool.py",
    "SRC/Agent/agent.py",
    "Model/fraud_xgboost_model.joblib"
]

for file in required_files:
    file_path = PROJECT_ROOT / file
    status = "FOUND" if file_path.exists() else "MISSING"
    print(f"{status}: {file}")

FOUND: SRC/Agent/prediction_tool.py
FOUND: SRC/Agent/explanation_tool.py
FOUND: SRC/Agent/agent.py
FOUND: Model/fraud_xgboost_model.joblib


In [9]:
import sys

# Add the Agent folder to Python's import path
AGENT_PATH = PROJECT_ROOT / "SRC" / "Agent"

if str(AGENT_PATH) not in sys.path:
    sys.path.insert(0, str(AGENT_PATH))

print("Agent import path configured:", AGENT_PATH)

Agent import path configured: /content/drive/MyDrive/Colab Notebooks/INFS 348 Final Project/INFS-348-Final-Project/SRC/Agent


In [10]:
# Import the agent
from agent import analyze_transaction

print("Agent imported successfully!")

Agent imported successfully!


In [11]:
import pandas as pd

# Define dataset location
DATA_PATH = PROJECT_ROOT /"Data/credit_card_fraud_2026.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["is_fraud"].value_counts())

Dataset loaded successfully!
Dataset shape: (20000, 26)

Target distribution:
is_fraud
0    19661
1      339
Name: count, dtype: int64


In [13]:
from prediction_tool import feature_columns

print("Number of model features:", len(feature_columns))
print("Features:", feature_columns)

Number of model features: 24
Features: ['amount_usd', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'is_ai_generated_scam_attempt', 'merchant_risk_score', 'prior_disputes']


In [23]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(PROJECT_ROOT / "Requirements.txt")
])

0

In [27]:
# Select the first known fraudulent transaction
fraud_transaction = df[df["is_fraud"] == 1].iloc[0]

# Extract the 24 model features
transaction_data = {
    feature: fraud_transaction[feature]
    for feature in feature_columns
}

# Run the AI agent
fraud_result = analyze_transaction(transaction_data)

# Display results
print("Actual Class: Fraudulent")
print("Agent Prediction:", fraud_result["prediction"])
print("Fraud Probability:", fraud_result["fraud_probability"])
print("Classification Threshold:", fraud_result["classification_threshold"])

print("\nAI Explanation:")
print(fraud_result["explanation"].replace(". ", ".\n"))

Actual Class: Fraudulent
Agent Prediction: Legitimate
Fraud Probability: 0.6826
Classification Threshold: 0.6937450766563416

AI Explanation:
The model classifies this transaction as **Legitimate** because its estimated fraud probability is 68.26%, below the 69.37% classification threshold.
The merchant risk score and CVV retry count increase the estimated likelihood of fraud.
The velocity score, customer age, and IP-country mismatch decrease it.
This prediction is an estimate, not proof that the transaction is or is not fraudulent.


In [26]:
# Select the first known legitimate transaction
legitimate_transaction = df[df["is_fraud"] == 0].iloc[0]

# Extract the 24 model features
transaction_data = {
    feature: legitimate_transaction[feature]
    for feature in feature_columns
}

# Run the AI agent
legitimate_result = analyze_transaction(transaction_data)

# Display results
print("Actual Class: Legitimate")
print("Agent Prediction:", legitimate_result["prediction"])
print("Fraud Probability:", legitimate_result["fraud_probability"])
print("Classification Threshold:", legitimate_result["classification_threshold"])

print("\nAI Explanation:")
print(legitimate_result["explanation"].replace(". ", ".\n"))

Actual Class: Legitimate
Agent Prediction: Legitimate
Fraud Probability: 0.0451
Classification Threshold: 0.6937450766563416

AI Explanation:
The model classifies this transaction as **Legitimate**, with an estimated fraud probability of **4.51%**, below the classification threshold of **69.37%**.
The strongest listed contributors all decrease fraud likelihood: `velocity_score`, `cvv_retry_count`, `ip_country_mismatch`, `is_new_merchant`, and `auth_method_No Authentication`.
These are model contributions, not proof that the transaction is legitimate or fraudulent; the prediction is an estimate.


In [30]:
# Randomly select 3 fraudulent and 3 legitimate transactions
random_fraud = df[df["is_fraud"] == 1].sample(n=5, random_state=42)
random_legitimate = df[df["is_fraud"] == 0].sample(n=5, random_state=42)

random_transactions = pd.concat([
    random_fraud,
    random_legitimate
]).sample(frac=1, random_state=42)

for _, row in random_transactions.iterrows():

    transaction_data = {
        feature: row[feature]
        for feature in feature_columns
    }

    result = analyze_transaction(transaction_data)

    print(f"\nTransaction ID: {row['transaction_id']}")
    print(f"Actual Class: {'Fraudulent' if row['is_fraud'] == 1 else 'Legitimate'}")
    print(f"Agent Prediction: {result['prediction']}")
    print(f"Fraud Probability: {result['fraud_probability']:.2%}")
    print(f"AI Explanation: {result['explanation']}")
    print("-" * 60)


Transaction ID: 5563
Actual Class: Legitimate
Agent Prediction: Legitimate
Fraud Probability: 12.27%
AI Explanation: The model classified this transaction as **Legitimate**, with an estimated fraud probability of **12.27%**, below the **69.37%** classification threshold. The `numerical__used_vpn` and `numerical__cvv_retry_count` features increased the model’s fraud likelihood. The `numerical__velocity_score`, `numerical__merchant_risk_score`, and `categorical__auth_method_PIN` features decreased it. This prediction is an estimate based on the model, not proof that the transaction is or is not fraudulent.
------------------------------------------------------------

Transaction ID: 6704
Actual Class: Fraudulent
Agent Prediction: Fraudulent
Fraud Probability: 78.53%
AI Explanation: The model classifies this transaction as **fraudulent**, with an estimated fraud probability of **78.53%**, above the classification threshold of **69.37%**. The strongest factors increasing fraud likelihood 